In [1]:
# Pre-training SoRL on arithmatic generalization dataset
# ------------------------------------------------------ 
import torch
from sorl.gat_dream import GAT, GATConfig
torch.set_float32_matmul_precision('high')  # Enable TF32 for ~2x speedup

BOS_TOKEN_ID = 20
gat_config = GATConfig(
    vocab_sizes=[BOS_TOKEN_ID+1, 16],  # 16 abstract tokens
    n_layer=4,
    n_head=4,
    n_embd=128,
    device="cuda" if torch.cuda.is_available() else "cpu",
    bos_token_id=BOS_TOKEN_ID
)
    
model = GAT(gat_config)
# model = model.to("cuda")
# model = torch.compile(model)

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Generate multiplication data: 

```python data/arithmetic.py```

In [2]:
from sorl.arithmetic import DigitTokenizer

enc = DigitTokenizer()

enc.encode_multiplication(7, 12, 84, a_digits=3, b_digits=3, c_digits=6)

[0, 10, 10, 17, 4, 2, 4, 10, 11, 12, 4, 3, 4, 10, 10, 10, 10, 18, 14, 1]

In [3]:
# ---- Arithmatic generalization dataset loader ----
from sorl.arithmetic import data_generator
from sorl.arithmetic import DigitTokenizer

# --- tokenizer ---
tokenizer = DigitTokenizer()

# --- data loader ---
train_loader = data_generator(filename_pattern="data/multiplication/multiplication_train.bin", sequence_length=256, device="cpu")
val_loader = data_generator(filename_pattern="data/multiplication/multiplication_val_id.bin", sequence_length=64, device="cpu")

In [4]:
tokens = next(train_loader)
tokens

RuntimeError: generator raised StopIteration

In [13]:
# Cyclic training with compression & memorization stages
from sorl.dream_utils import sorl_evaluate_v2, sorl_search
from sorl.topo import orthogonalize_abs_param
from collections import defaultdict
from sorl.info import SoRLLoss_v9, SoRLLoss_v10

# --- orthogonal initialization on abs param --- 
orthogonalize_abs_param(model, do_wte=True, do_head=True)

# mem_loss_fn = SoRLLoss_v9(model.vocab_sizes[1], decay=0.8, target_vocab_util=0.9)
comp_loss_fn = SoRLLoss_v10(model.vocab_sizes[1], decay=0.8, target_vocab_util=0.9)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
n = 2
temperature = torch.tensor([0.0, 5.0], device=model.device)
num_steps = 400
alpha_abs = 0.1
alpha_soft_zipf = 1.0
alpha_info_gain = 10.0
phase = "compression"

record = defaultdict(list)
img_frames = []

# Just the most biologically plausible setting, sensor input is lost, once encoded into neuron state
memory_span_abs = K 
memory_span_traj = K

for step in range(num_steps): 

    optimizer.zero_grad()

    tokens, doc_ids = train_loader.get_batch(batch_size)

    with torch.no_grad(): 
        best_data, best_traj_ppt, best_abs_ppt, search_adv = sorl_search(tokens, model, n=n, K=K, max_iterations=max_iterations, memory_span_abs=memory_span_abs, memory_span_traj=memory_span_traj, attn_blocksize=attn_blocksize, temperature=temperature, truncate_seq_len=False)

    # --- compute loss --- 
    cond_traj_loss, abs_loss, zipf_bigram_loss = comp_loss_fn(best_data, model, memory_span_abs, memory_span_traj, attn_blocksize)
    loss = cond_traj_loss + alpha_abs * abs_loss + alpha_soft_zipf * zipf_bigram_loss

    # --- log relative info gain ---
    base_traj_loss = model.forward(tokens, memory_span_abs, memory_span_traj, attn_blocksize)[0].mean()
    rel_info_gain = (base_traj_loss - cond_traj_loss) / base_traj_loss

    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 2 == 0: 
        with torch.no_grad(): 
            temperatures_eval = torch.tensor([0.0, 10.0], device=model.device)

            val_tokens, val_adv, traj_loss, abs_loss, abs_logits, abs_tokens, avg_logit_sim = sorl_evaluate_v2(tokens, model, n=2, K=K, max_iterations=max_iterations, memory_span_abs=memory_span_abs, memory_span_traj=memory_span_traj, attn_blocksize=attn_blocksize, temperature=temperatures_eval,
                                                                     truncate_seq_len=False)
            _, _, zipf_bigram_loss = comp_loss_fn(val_tokens, model, memory_span_abs, memory_span_traj, attn_blocksize)
     
            abs_stats.update(abs_logits, traj_loss, rel_info_gain, abs_tokens, doc_ids)
            record['vocab_util'].append(abs_stats.vocab_util * 100)
            record['greedy_adv'].append(val_adv.item() * 100)
            record['abs_loss'].append(abs_loss.mean().item())
            record['traj_loss'].append(traj_loss.mean().item())
            record['base_traj_loss'].append(base_traj_loss.item())
            record['bigram_rep_rate'].append(abs_stats.bigram_rep_rate)
            record['kl_soft_zipf'].append(zipf_bigram_loss.item())
            record['rel_search_info_gain'].append(rel_info_gain)

        print(f"\n{phase} | step {step} | base traj loss: {base_traj_loss.item():.2f} | cond traj loss: {traj_loss.mean().item():.2f} | rel search info gain: {rel_info_gain * 100:.2f}% | greedy adv: {val_adv.item() * 100:.2f}% | vocab util: {abs_stats.vocab_util * 100:.2f}%  | avg logit sim: {avg_logit_sim:.2f} |  bigram-zipf kl: {zipf_bigram_loss.item():.2f} | bigram rep rate: {abs_stats.bigram_rep_rate:.2f} | rel info gain (search): {rel_info_gain:.2f}")
        
        img = visualize_dynamics(abs_stats, loader, model, enc, K, step)
        img_frames.append(img)
        # break

AttributeError: 'generator' object has no attribute 'get_batch'

In [11]:
from sorl.neo_utils import sorl_search, sorl_search, sorl_evaluate
from sorl.info import SoRLLoss_v9
from sorl.gapt import GatedPhaseTransition

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)

# memory_span = 1
memory_span = 25 # control amount of memory stored in trajectory token (rest is stored in abstraction token)
attn_blocksize = 1792
K = 4  # abstraction ratio
n = 2  # number of rollout
temperatures = torch.tensor([0.5, 10.0], device=model.device)
max_iterations = 2  # of denoising steps for generating abstraction token (in parallel)
alpha_loss = 0.1 # weight on abstraction perplexity (training)
alpha_sel = 0.2  # weight on curiosity reward (during selection)
sel_mode = "vocab_util" # selection mode (abs_ppt / vocab_util)
n_eval = 4  # number of rollout for evaluation
temperatures_eval = torch.tensor([0.5, 5.0, 5.0, 5.0], device=model.device) # temperature for evaluation
loss_fn = SoRLLoss_v9(model.vocab_sizes[0])

num_steps = 500
gapt = GatedPhaseTransition(p_m=10)

for step in range(num_steps): 
 
    optimizer.zero_grad()

    tokens = next(train_loader)

    # --- mixture of SoRL selection & deep supervision (avg. loss per iteration) ---
    with torch.no_grad(): 
        search_tokens, search_ppt, search_adv = sorl_search_v2(tokens, model, n=n, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperatures,
                                                               truncate_seq_len=False, alpha_select=alpha_sel, select_mode=sel_mode)
    
    # --- compute loss ---
    cond_traj_loss, abs_loss, zipf_bigram_loss = loss_fn(best_data, model, memory_span_abs, memory_span_traj, attn_blocksize)
    loss = cond_traj_loss + alpha_abs * abs_loss + alpha_soft_zipf * zipf_bigram_loss

    # --- log relative info gain ---
    base_traj_loss = model.forward(tokens, memory_span_abs, memory_span_traj, attn_blocksize)[0].mean()
    rel_info_gain = (base_traj_loss - cond_traj_loss) / base_traj_loss
                                       
    loss = gapt.step(traj_loss, alpha_loss * abs_loss, verbose=True) # gated phase transition loss adaptor

    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 2 == 0: 
        with torch.no_grad(): 
            val_tokens, val_adv, traj_loss, abs_loss = sorl_evaluate(search_tokens, model, n=n_eval, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperatures_eval,
                                                                   truncate_seq_len=False)
            traj_loss, abs_loss = compute_loss(search_tokens, model, memory_span=memory_span, attn_blocksize=attn_blocksize)
            # vocab_util = compute_vocab_utilization_rate(val_tokens, model)
            vocab_util = 0.0
        print(f"validation step {step} | traj_loss: {traj_loss.item():.2f} | abs_loss: {abs_loss.item():.2f} | search adv: {val_adv.item() * 100:.2f}% | vocab util: {vocab_util * 100:.2f}%")

RuntimeError: generator raised StopIteration

In [4]:
# generate function implementation 
# ----------------------------------
from sorl.neo_utils import generate
from sorl.arithmetic import process_query, check_answer

K = 5
tokens = next(val_loader)
idx, answer_idx = process_query(tokens)

print(f"init   | idx: {idx[0].tolist()} | question: {tokenizer.decode(idx[0].tolist()[1:-1])}")
for i in range(len(answer_idx[0])*2): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, 
                   temperature=temperatures_eval[0])
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    print(f"step {i+1} idx (abstraction free): {idx_without_abstraction.tolist()}")
    print(f"                         idx : {idx[0].tolist()}")

is_correct, pred_answer, true_answer = check_answer(idx_without_abstraction, answer_idx, tokenizer)
print(f"is_correct: {is_correct} | pred_answer: {pred_answer} | true_answer: {true_answer}")

init   | idx: [0, 15, 4, 2, 4, 18, 4, 3] | question: 5 x 8 
step 1 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4]
                         idx : [0, 15, 4, 2, 4, 25, 18, 4, 3, 4]
step 2 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 13]
                         idx : [0, 15, 4, 2, 4, 24, 18, 4, 3, 4, 13]
step 3 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 13, 10]
                         idx : [0, 15, 4, 2, 4, 21, 18, 4, 3, 4, 13, 24, 10]
step 4 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 13, 10, 1]
                         idx : [0, 15, 4, 2, 4, 21, 18, 4, 3, 4, 13, 23, 10, 1]
step 5 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 13, 10, 1, 0]
                         idx : [0, 15, 4, 2, 4, 23, 18, 4, 3, 4, 13, 24, 10, 1, 0]
step 6 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 13, 10, 1, 0, 16]
                         idx : [0, 15, 4, 2, 4, 24, 18, 4, 3, 4, 13, 23, 10, 1, 0, 16]
step 7 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 